In [ ]:
import requests


url = "https://api.divar.ir/v8/postlist/w/search"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Content-Type': 'application/json',
    'Referer': 'https://divar.ir/'
}

payload = {
    "city_ids": ["1"], 
    "search_data": {
        "form_data": {
            "data": {
                "category": {"str": {"value": "real-estate"}}
            }
        }
    }
}

print("Sending request to Divar...")
response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    print("Success! Status 200")
    data = response.json()
    
    print("Keys in response:", data.keys())
    
    if 'pagination' in data:
        print("\nNext page info:", data['pagination'])
        
else:
    print("Error:", response.status_code)
    print(response.text)


In [ ]:
import requests
import pandas as pd
import os
import time
import random


search_url = "https://api.divar.ir/v8/postlist/w/search"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Content-Type": "application/json"
}

all_ads = []
last_post_date = ""
page = 1
max_pages = 450

while page <= max_pages:
    print(f"Page : {page}...")
    
    payload = {
        "city_ids": ["1"],
        "search_data": {
            "form_data": {
                "data": {
                    "category": {"str": {"value": "residential-sell"}}
                }
            }
        }
    }
    
    if last_post_date:
        payload["pagination_data"] = {
            "@type": "type.googleapis.com/post_list.PaginationData",
            "last_post_date": last_post_date,
            "page": page,
            "layer_page": page
        }

    try:
        response = requests.post(search_url, json=payload, headers=headers)
        if response.status_code == 200:
            data = response.json()
            widgets = data.get("list_widgets", [])
            
            for widget in widgets:
                if widget.get("widget_type") == "POST_ROW":
                    post_data = widget.get("data", {})
                    token = post_data.get("token")
                    district = post_data.get("action", {}).get("payload", {}).get("web_info", {}).get("district_persian", "Nan")
                    
                    if token:
                        all_ads.append({
                            "token": token,
                            "district": district
                        })
            
            pagination = data.get("pagination", {})
            last_post_date = pagination.get("data", {}).get("last_post_date")
            
            if page % 50 == 0:
                pd.DataFrame(all_ads).to_csv("divar_tokens_backup.csv", index=False)
                print(f"--- Backup saved at page {page} ---")

            
            if not last_post_date:
                break
            page += 1
            time.sleep(random.uniform(1,4))
        else:
            break
    except Exception as e:
        print(f"Phase 1 Error: {e}. Retrying in 10 seconds...")
        time.sleep(10)



In [ ]:
import requests
import time
import random
import pandas as pd
import os

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Content-Type": "application/json"
}

input_tokens_file = "data/raw/divar_home_tokens.csv"
all_ads = []

if os.path.exists(input_tokens_file):
    try:
        df_tokens = pd.read_csv(input_tokens_file)
        required_cols = ['token', 'district']
        
        if not all(col in df_tokens.columns for col in required_cols):
            print(f"Error: Missing required columns ({required_cols}) in {input_tokens_file}. Please ensure 'token' and 'district' columns exist.")
            exit() 

        all_ads = df_tokens[required_cols].to_dict(orient='records')
        print(f"Loaded {len(all_ads)} ad tokens from {input_tokens_file}.")
    except Exception as e:
        print(f"Error loading tokens from {input_tokens_file}: {e}")
        exit()
else:
    print(f"Error: Token file '{input_tokens_file}' not found. Please ensure it exists in 'data/raw/'.")
    exit()

if not all_ads:
    print("No ad tokens available to process. Exiting detail extraction.")
    exit()
else:
    print("Starting detail extraction phase...")

detailed_data = []
checkpoint_size = 300
output_file = "divar_full_data.csv"

existing_tokens = set()
if os.path.exists(output_file):
    try:
        existing_df = pd.read_csv(output_file, usecols=['Token'])
        existing_tokens = set(existing_df['Token'].unique())
        print(f"Found {len(existing_tokens)} existing records in {output_file}. Will skip these.")
    except Exception as e:
        print(f"Could not read existing data from {output_file}: {e}. Starting fresh.")

for index, ad in enumerate(all_ads, 1):
    token = ad.get('token')
    district = ad.get('district')
    
    
    
    if not token or not district:
        print(f"Skipping ad at index {index} due to missing token or district.")
        continue
    
    if token in existing_tokens:
        continue
    
    print(f"[*] Processing index: {index} / {len(all_ads)} | Token: {token}")

    detail_url = f"https://api.divar.ir/v8/posts-v2/web/{token}"

    try:
        res = requests.get(detail_url, headers=headers,timeout=25)
        
        if res.status_code == 200:
            detail_json = res.json()
            area, year, rooms, exact_price, floor, parking, storage, elevator = [None]*8
            
            webengage = detail_json.get("webengage", {})
            exact_price = webengage.get("price")
            
            sections = detail_json.get("sections", [])
            for section in sections:
                if section.get("section_name") == "LIST_DATA":
                    widgets = section.get("widgets", [])
                    for widget in widgets:
                        widget_type = widget.get("widget_type")
                        data = widget.get("data", {})

                        if widget_type == "GROUP_INFO_ROW":
                            for item in data.get("items", []):
                                item_title = item.get("title", "")
                                item_value = item.get("value", "")
                                if "متراژ" in item_title: area = item_value
                                elif "ساخت" in item_title: year = item_value
                                elif "اتاق" in item_title: rooms = item_value
                                    
                        elif widget_type == "UNEXPANDABLE_ROW":
                            if data.get("title") == "طبقه":
                                floor = data.get("value")
                        elif widget_type == "GROUP_FEATURE_ROW":
                            for item in data.get("items", []):
                                item_title = item.get("title", "")
                                if "پارکینگ" in item_title:
                                    parking = item.get("available", False)
                                elif "انباری" in item_title:
                                    storage = item.get("available", False)
                                elif "آسانسور" in item_title:
                                    elevator = item.get("available", False)
                                    
            detailed_data.append({
                "Token": token, "District": district, "Area": area, "Year": year,
                "Rooms": rooms, "Floor": floor, "Parking": parking,
                "Storage": storage, "Elevator": elevator, "Exact_Price": exact_price
            })
            
        else:
            print(f"Failed to fetch details for token {token}. Status code: {res.status_code}. Response: {res.text[:150]}...")
        
        if len(detailed_data) >= checkpoint_size:
            df = pd.DataFrame(detailed_data)
            if not os.path.exists(output_file):
                df.to_csv(output_file, index=False, encoding='utf-8-sig')
            else:
                df.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8-sig')
            existing_tokens.update(df['Token'].tolist())
            print(f"Saved {len(detailed_data)} new records. Total unique processed: {len(existing_tokens)}.")
            detailed_data = []
        
        time.sleep(random.uniform(1, 4))
        
    except requests.exceptions.RequestException as e:
        print(f"Network error or request issue for token {token}: {e}. Skipping to next.")
        continue
    except Exception as e:
        print(f"An unexpected error occurred for token {token}: {e}. Skipping to next.")
        continue

if detailed_data:
    df = pd.DataFrame(detailed_data)
    if not os.path.exists(output_file):
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
    else:
        df.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8-sig')
    print(f"Final {len(detailed_data)} records saved to {output_file}.")

print("Detail extraction phase completed.")
